In [ ]:
!pip install -q transformers datasets accelerate sacrebleu evaluate sentencepiece
!pip install -q torch
!git clone https://github.com/stefan-it/nmt-en-vi.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
Cloning into 'nmt-en-vi'...
remote: Enumerating objects: 57, done.
remote: Total 57 (delta 0), reused 0 (delta 0), pack-reused 57 (from 1)
Receiving objects: 100% (57/57), 9.82 MiB | 38.24 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [ ]:
import os
import tarfile
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer
data_dir = "nmt-en-vi/data"
tgz_files = [
    "train-en-vi.tgz",
    "test-2013-en-vi.tgz"
]

for tgz_name in tgz_files:
    file_path = os.path.join(data_dir, tgz_name)
    if os.path.exists(file_path):
        try:
            with tarfile.open(file_path, "r:gz") as tar:
                tar.extractall(path=data_dir)
        except Exception as e:
            print(f"Lỗi {tgz_name}: {e}")
print(os.listdir(data_dir))
def read_text_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines()]
train_en_path = os.path.join(data_dir, "train.en")
train_vi_path = os.path.join(data_dir, "train.vi")
test_en_path = os.path.join(data_dir, "tst2013.en")
test_vi_path = os.path.join(data_dir, "tst2013.vi")
train_en = read_text_file(train_en_path)
train_vi = read_text_file(train_vi_path)
test_en = read_text_file(test_en_path)
test_vi = read_text_file(test_vi_path)
train_dataset = Dataset.from_dict({"en": train_en, "vi": train_vi})
test_dataset = Dataset.from_dict({"en": test_en, "vi": test_vi})
raw_datasets = DatasetDict({"train": train_dataset, "test": test_dataset})
model_checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
prefix = "translate English to Vietnamese: "
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [prefix + ex for ex in examples["en"]]
    targets = examples["vi"]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)


/tmp/ipython-input-476265508.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=data_dir)


['tst2013.en', 'train.en', 'train-en-vi.tgz', 'dev-2012-en-vi.tgz', 'train.vi', 'test-2013-en-vi.tgz', 'tst2013.vi']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Map:   0%|          | 0/133317 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

In [ ]:
drive.mount('/content/drive')
MODEL_NAME = "google/t5-efficient-tiny"
DRIVE_FINAL_DIR = "/content/drive/MyDrive/Models/t5-tiny-fair-compare-v3_final"
LOCAL_WORK_DIR = "/content/t5_tiny_work_v3_final"
WANDB_PROJECT_NAME = "t5-tiny-fair-compare-v3_final"

if not os.path.exists(DRIVE_FINAL_DIR):
    os.makedirs(DRIVE_FINAL_DIR)
try:
    wandb.finish()
except:
    pass

WANDB_ID_FILE = os.path.join(DRIVE_FINAL_DIR, "wandb_id.json")
run_id = None

if os.path.exists(WANDB_ID_FILE):
    with open(WANDB_ID_FILE, "r") as f:
        run_id = json.load(f)["run_id"]
else:
    run_id = wandb.util.generate_id()
    with open(WANDB_ID_FILE, "w") as f:
        json.dump({"run_id": run_id}, f)

os.environ["WANDB_PROJECT"] = WANDB_PROJECT_NAME
os.environ["WANDB_RUN_ID"] = run_id
os.environ["WANDB_RESUME"] = "allow"
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
prefix = "translate English to Vietnamese: "

def preprocess_function(examples):
    inputs = [prefix + ex for ex in examples["en"]]
    targets = examples["vi"]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    labels = tokenizer(text_target=targets, max_length=128, truncation=True)
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs
if 'raw_datasets' in globals():
    tokenized_datasets = raw_datasets.map(preprocess_function, batched=True, desc="Processing data")
else:
    raise ValueError("⚠️ Biến 'raw_datasets' chưa được load. Hãy chạy lại cell tải dữ liệu ban đầu.")

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
class ZipAndBackupCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        if state.best_model_checkpoint:
             ckpt_path = state.best_model_checkpoint
        else:
             ckpt_path = os.path.join(args.output_dir, f"checkpoint-{state.global_step}")

        if os.path.exists(ckpt_path):
            folder_name = os.path.basename(ckpt_path)
            shutil.make_archive(ckpt_path, 'zip', ckpt_path)
            local_zip_path = ckpt_path + ".zip"
            drive_dest_zip = os.path.join(DRIVE_FINAL_DIR, folder_name + ".zip")

            try:
                if os.path.exists(drive_dest_zip):
                    os.remove(drive_dest_zip)
                shutil.copy2(local_zip_path, drive_dest_zip)
                drive_dest_folder = os.path.join(DRIVE_FINAL_DIR, folder_name)
                if os.path.exists(drive_dest_folder):
                    shutil.rmtree(drive_dest_folder)
                shutil.copytree(ckpt_path, drive_dest_folder)
            except Exception:
                pass
args = Seq2SeqTrainingArguments(
    output_dir=LOCAL_WORK_DIR,
    report_to="wandb",
    run_name="t5-tiny-clean-run",
    fp16=False,
    optim="adafactor",
    learning_rate=1e-3,
    save_strategy="steps",
    save_steps=1000,
    eval_strategy="steps",
    eval_steps=1000,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    num_train_epochs=20,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    predict_with_generate=True,
    gradient_checkpointing=False,
    dataloader_pin_memory=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=10),
        ZipAndBackupCallback()
    ]
)
resume_path = None
last_checkpoint_local = get_last_checkpoint(LOCAL_WORK_DIR)

if last_checkpoint_local:
    resume_path = last_checkpoint_local
else:
    if os.path.exists(DRIVE_FINAL_DIR):
        ckpts = [d for d in os.listdir(DRIVE_FINAL_DIR) if d.startswith("checkpoint-")]
        if ckpts:
            ckpts.sort(key=lambda x: int(x.split("-")[-1]), reverse=True)
            for ckpt_name in ckpts:
                drive_src = os.path.join(DRIVE_FINAL_DIR, ckpt_name)
                if os.path.exists(os.path.join(drive_src, "pytorch_model.bin")) or os.path.exists(os.path.join(drive_src, "model.safetensors")):
                    local_dest = os.path.join(LOCAL_WORK_DIR, ckpt_name)
                    if os.path.exists(local_dest): shutil.rmtree(local_dest)
                    shutil.copytree(drive_src, local_dest)
                    resume_path = local_dest
                    break

if resume_path:
    trainer.train(resume_from_checkpoint=resume_path)
else:
    trainer.train()

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23020412 (23020412-vnu-uet) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/62.3M [00:00<?, ?B/s]

Processing data:   0%|          | 0/1268 [00:00<?, ? examples/s]

/tmp/ipython-input-2137330409.py:118: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss
49000,0.910600,0.813682
50000,0.909800,0.811314
51000,0.903900,0.812370
52000,0.904900,0.808204
53000,0.899000,0.808060
54000,0.898700,0.804890
55000,0.895900,0.805146
56000,0.890800,0.801373
57000,0.893700,0.797713
58000,0.891600,0.796443


wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-49000)... Done. 0.3s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-50000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-51000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-52000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-53000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-54000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-55000)... Done. 0.2s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-56000)... Done. 0.3s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-57000)... Done. 0.3s
wandb: Adding directory to artifact (/content/t5_tiny_work_v3_final/checkpoint-58000)... Done. 0.2s


In [9]:
from google.colab import drive
import os
import tarfile
import zipfile
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
import torch
import math
import numpy as np
import evaluate
import random
import shutil

drive.mount('/content/drive')
DRIVE_FINAL_DIR = "/content/drive/MyDrive/Models/t5-tiny-fair-compare-v3_final"
TEMP_UNZIP_DIR = "/content/temp_checkpoint_eval"


all_items = os.listdir(DRIVE_FINAL_DIR)
checkpoints = [x for x in all_items if "checkpoint-" in x]

def get_step(name):
    clean_name = name.replace(".zip", "").replace("checkpoint-", "")
    return int(clean_name)

latest_ckpt_name = sorted(checkpoints, key=get_step)[-1]
drive_src_path = os.path.join(DRIVE_FINAL_DIR, latest_ckpt_name)


FINAL_CHECKPOINT_PATH = ""
if os.path.isdir(drive_src_path):
    FINAL_CHECKPOINT_PATH = drive_src_path
elif latest_ckpt_name.endswith(".zip"):
    if os.path.exists(TEMP_UNZIP_DIR):
        shutil.rmtree(TEMP_UNZIP_DIR)
    os.makedirs(TEMP_UNZIP_DIR)

    with zipfile.ZipFile(drive_src_path, 'r') as zip_ref:
        zip_ref.extractall(TEMP_UNZIP_DIR)
    FINAL_CHECKPOINT_PATH = TEMP_UNZIP_DIR

try:
    tokenizer = AutoTokenizer.from_pretrained(FINAL_CHECKPOINT_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_CHECKPOINT_PATH).to("cuda" if torch.cuda.is_available() else "cpu")
except Exception as e:
    raise

if 'raw_datasets' not in globals():
    if not os.path.exists("nmt-en-vi"):
        os.system("git clone https://github.com/stefan-it/nmt-en-vi.git")

    def read_text_file(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            return [line.strip() for f in f.readlines()]

    data_dir = "nmt-en-vi/data"
    for f in ["test-2013-en-vi.tgz"]:
        path = os.path.join(data_dir, f)
        if os.path.exists(path):
            with tarfile.open(path, "r:gz") as tar: tar.extractall(path=data_dir)

    test_en = read_text_file(os.path.join(data_dir, "tst2013.en"))
    test_vi = read_text_file(os.path.join(data_dir, "tst2013.vi"))
    raw_test_dataset = Dataset.from_dict({"en": test_en, "vi": test_vi})
else:
    raw_test_dataset = raw_datasets["test"]

prefix = "translate English to Vietnamese: "

def preprocess_test(examples):
    inputs = [prefix + ex for ex in examples["en"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["vi"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

test_dataset_processed = raw_test_dataset.map(preprocess_test, batched=True)

metric = evaluate.load("sacrebleu")
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

eval_args = Seq2SeqTrainingArguments(
    output_dir="temp_eval_results",
    per_device_eval_batch_size=16,
    predict_with_generate=True,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    tokenizer=tokenizer,
    eval_dataset=test_dataset_processed
)



eval_metrics = trainer.evaluate()
current_loss = eval_metrics["eval_loss"]
try:
    perplexity = math.exp(current_loss)
except OverflowError:
    perplexity = float('inf')

predict_results = trainer.predict(test_dataset_processed, metric_key_prefix="test", max_length=128)
predictions = predict_results.predictions
labels = predict_results.label_ids


predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

decoded_preds = [pred.strip() for pred in decoded_preds]
decoded_labels_formatted = [[label.strip()] for label in decoded_labels]

bleu_score = metric.compute(predictions=decoded_preds, references=decoded_labels_formatted)




print(f" Validation Loss : {current_loss:.4f}")
print(f" Perplexity (PPL): {perplexity:.2f} ")
print(f" BLEU Score      : {bleu_score['score']:.2f}")

print("\n Demo dịch")
indices = random.sample(range(len(decoded_preds)), 5)
for i in indices:
    print(f"🇬🇧 Input : {raw_test_dataset[i]['en']}")
    print(f"🇻🇳 Ref   : {decoded_labels[i]}")
    print(f"Model : {decoded_preds[i]}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-1025261538.py:91: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


 Validation Loss : 0.7758
 Perplexity (PPL): 2.17 
 BLEU Score      : 25.65

 Demo dịch
🇬🇧 Input : And he said that he needed those guns because of the trauma he &apos;d experienced as a young boy .
🇻🇳 Ref   : Và anh ta nói rng anh ta cn nhng cây sng này bi v nhng tn thng mà anh  tri qua trong quá kh khi là mt a tr .
Model : Và ông y nói rng ông cn nhng sng sót này bi v s tri nghim mà ông y  tri qua nh mt cu bé .
🇬🇧 Input : Am I South Korean or North Korean ?
🇻🇳 Ref   : Tôi là ngi Nam Triu Tiên hay Bc Triu Tiên ?
Model : Liu tôi có phi Nam hay Bc  ?
🇬🇧 Input : And so just as the womb entirely envelopes the embryo , which grows within it , the divine matrix of compassion nourishes the entire existence .
🇻🇳 Ref   : Và bi v t cung bao bc hoàn toàn phôi thai ang phát trin trong lng nó , ma trn thiêng liêng ca tnh thng nuôi dng toàn b s sng ó .
Model : Và v vy , khi cái mt cái t c hoàn toàn to ra các th trng , mà tăng trng trong nó , mt ma trn tuyt vi ca lng trc ng tn tn tng
🇬🇧 Input : It &

In [12]:
import torch
import math
import numpy as np
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    MBart50TokenizerFast,
    MBartForConditionalGeneration
)

metric = evaluate.load("sacrebleu")
device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate_model_pipeline(model_checkpoint, test_dataset, prefix="", model_type="auto"):
    if model_type == "mbart":
        tokenizer = MBart50TokenizerFast.from_pretrained(model_checkpoint)
        model = MBartForConditionalGeneration.from_pretrained(model_checkpoint).to(device)
        tokenizer.src_lang = "en_XX"
        tokenizer.tgt_lang = "vi_VN"
    else:
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)
    def preprocess_eval(examples):
        inputs = [prefix + ex for ex in examples["en"]]
        targets = examples["vi"]
        if model_type == "mbart":
            model_inputs = tokenizer(inputs, max_length=128, truncation=True)
        else:
            model_inputs = tokenizer(inputs, max_length=128, truncation=True)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(targets, max_length=128, truncation=True)

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    eval_set = test_dataset.map(preprocess_eval, batched=True, remove_columns=test_dataset.column_names)
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    args = Seq2SeqTrainingArguments(
        output_dir=f"./eval_logs_{model_checkpoint.replace('/', '_')}",
        per_device_eval_batch_size=16,
        predict_with_generate=True,
        fp16=True if device == "cuda" else False,
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        data_collator=data_collator,
        tokenizer=tokenizer,
    )
    eval_metrics = trainer.evaluate(eval_dataset=eval_set)
    try:
        perplexity = math.exp(eval_metrics["eval_loss"])
    except OverflowError:
        perplexity = float("inf")
    gen_kwargs = {}
    if model_type == "mbart":
        gen_kwargs = {"forced_bos_token_id": tokenizer.lang_code_to_id["vi_VN"]}

    predict_results = trainer.predict(
        eval_set,
        metric_key_prefix="predict",
        max_length=128,
        **gen_kwargs
    )

    preds = predict_results.predictions
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = [[ref.strip()] for ref in test_dataset["vi"]]
    decoded_preds = [pred.strip() for pred in decoded_preds]

    bleu_result = metric.compute(predictions=decoded_preds, references=decoded_labels)

    print(f"\n KẾT QUẢ CHO {model_checkpoint}:")
    print(f"Perplexity: {perplexity:.2f}")
    print(f"BLEU Score:       {bleu_result['score']:.2f}")
    print(f"Input: {test_dataset['en'][0]}")
    print(f"Model: {decoded_preds[0]}")
    print(f"Ref  : {decoded_labels[0][0]}")

In [13]:

evaluate_model_pipeline(
    model_checkpoint="Helsinki-NLP/opus-mt-en-vi",
    test_dataset=test_dataset,
    prefix="",
    model_type="auto"
)

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/809k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/289M [00:00<?, ?B/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-1454617230.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



 KẾT QUẢ CHO Helsinki-NLP/opus-mt-en-vi:
Perplexity: 7.39
BLEU Score:       29.85
Input: When I was little , I thought my country was the best on the planet , and I grew up singing a song called &quot; Nothing To Envy . &quot;
Model: Khi tôi còn nhỏ, tôi nghĩ rằng đất nước của tôi là nơi tốt nhất trên hành tinh này, và tôi lớn lên trong một bài hát tên là &quot; không có gì để &.
Ref  : Khi tôi còn nhỏ , Tôi nghĩ rằng BắcTriều Tiên là đất nước tốt nhất trên thế giới và tôi thường hát bài &quot; Chúng ta chẳng có gì phải ghen tị . &quot;


In [14]:

evaluate_model_pipeline(
    model_checkpoint="NlpHUST/t5-en-vi-small",
    test_dataset=test_dataset,
    prefix="translate English to Vietnamese: ",
    model_type="auto"
)

tokenizer_config.json:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-1454617230.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



 KẾT QUẢ CHO NlpHUST/t5-en-vi-small:
Perplexity: 2.46
BLEU Score:       25.79
Input: When I was little , I thought my country was the best on the planet , and I grew up singing a song called &quot; Nothing To Envy . &quot;
Model: Dịch tiếng Anh sang tiếng Việt: Khi tôi còn nhỏ , tôi nghĩ đất nước tôi là người giỏi nhất trên hành tinh này , và tôi lớn lên hát một bài hát gọi là " Không có gì để Envy " .
Ref  : Khi tôi còn nhỏ , Tôi nghĩ rằng BắcTriều Tiên là đất nước tốt nhất trên thế giới và tôi thường hát bài &quot; Chúng ta chẳng có gì phải ghen tị . &quot;


In [15]:
evaluate_model_pipeline(
    model_checkpoint="facebook/mbart-large-50-many-to-many-mmt",
    test_dataset=test_dataset,
    prefix="",
    model_type="mbart"
)

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Map:   0%|          | 0/1268 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4169: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/tmp/ipython-input-1454617230.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(



 KẾT QUẢ CHO facebook/mbart-large-50-many-to-many-mmt:
Perplexity: 5.66
BLEU Score:       34.56
Input: When I was little , I thought my country was the best on the planet , and I grew up singing a song called &quot; Nothing To Envy . &quot;
Model: Khi tôi còn nhỏ , tôi nghĩ rằng đất nước của tôi là tốt nhất trên hành tinh này , và tôi lớn lên hát một bài hát tên là &quot; Không có gì đáng ghen tị . &quot;
Ref  : Khi tôi còn nhỏ , Tôi nghĩ rằng BắcTriều Tiên là đất nước tốt nhất trên thế giới và tôi thường hát bài &quot; Chúng ta chẳng có gì phải ghen tị . &quot;
